# Fine-tune NomNaOCR CRNN×CTC trên chữ Sách Thánh Truyện

**Chuẩn bị (làm 1 lần):**
1. Add data → **Add Input** → upload `nomna_finetune_all.zip`.
2. **Settings → Accelerator → GPU P100** (hoặc T4 x2 cũng được — script dùng 1 GPU).
3. Run all. Kết quả ở **Output**: `finetuned_CRNNxCTC.h5` + `finetuned_vocab.txt`.
4. Cell 5 (tuỳ chọn): đẩy checkpoint lên HuggingFace — dán HF **WRITE** token.

> Repo dùng API Keras-2 (ctc_batch_cost/ctc_decode) nên ta bật `TF_USE_LEGACY_KERAS=1`.

In [ ]:
# 1) Environment: Keras-2 compat + GPU check
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
!pip install -q tf_keras 2>/dev/null
import tensorflow as tf
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
# 2) Locate uploaded files (robust: extracted OR still-zipped)
import glob, os, zipfile
hits = glob.glob('/kaggle/input/**/finetune_kaggle.py', recursive=True)
if not hits:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if zips:
        print('Đang giải nén', zips[0], '...')
        os.makedirs('/kaggle/working/nomna', exist_ok=True)
        zipfile.ZipFile(zips[0]).extractall('/kaggle/working/nomna')
        hits = glob.glob('/kaggle/working/nomna/**/finetune_kaggle.py', recursive=True)
if not hits:
    print('DEBUG /kaggle/input:')
    [print(' ', p) for p in glob.glob('/kaggle/input/**', recursive=True)[:60]]
    raise SystemExit('Không thấy finetune_kaggle.py — xem list trên.')
ROOT = os.path.dirname(hits[0])
print('ROOT =', ROOT)
print('Contents:', sorted(os.listdir(ROOT)))

In [ ]:
# 3) Fine-tune (transfer CNN+BiGRU, rebuild head to Sách vocab, train CTC)
!TF_USE_LEGACY_KERAS=1 python "{ROOT}/finetune_kaggle.py" \
    --dataset_dir "{ROOT}/Datasets/Patches" \
    --pretrained  "{ROOT}/NomNaOCR_CRNNxCTC.h5" \
    --repo        "{ROOT}/Text recognition" \
    --epochs 30 --batch 64 --out /kaggle/working

In [ ]:
# 4) Check outputs -> download these 2 files from the Output panel
!ls -la /kaggle/working/finetuned_CRNNxCTC.h5 /kaggle/working/finetuned_vocab.txt
print('\nTải 2 file trên về, đặt vào NomNaOCR/weights/, rồi chạy run_nomnaocr_consensus.py --chunk 8')

In [ ]:
# 5) (tuỳ chọn) Đẩy checkpoint lên HuggingFace — DÁN HF **WRITE** TOKEN
!pip install -q huggingface_hub
from huggingface_hub import HfApi
HF_TOKEN = "hf_xxx".strip()                 # <-- DÁN token loại WRITE: huggingface.co/settings/tokens
api = HfApi(token=HF_TOKEN)
me = api.whoami()                           # token sai -> lỗi rõ ngay ở đây
print("Đăng nhập:", me["name"])
REPO_ID = f'{me["name"]}/nomna-sach-crnn'   # tự lấy đúng username -> khỏi lệch namespace
api.create_repo(REPO_ID, repo_type="model", exist_ok=True)
for f in ["finetuned_CRNNxCTC.h5", "finetuned_vocab.txt"]:
    api.upload_file(path_or_fileobj=f"/kaggle/working/{f}",
                    path_in_repo=f, repo_id=REPO_ID, repo_type="model")
print("Xong ->", f"https://huggingface.co/{REPO_ID}")